# Baseline 2: Zero-Shot LLM via Groq API (Llama-3)

In this notebook, we evaluate a modern Large Language Model (LLM) without any task-specific fine-tuning (Zero-Shot). We use the `Llama-3-8b` model accessed via the Groq API.

**Objective:** Test whether a generalized LLM can natively distinguish between true toxicity and typical multiplayer game banter based solely on its pre-trained knowledge and a carefully engineered prompt.

In [4]:
import os
import pandas as pd
from groq import Groq
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm.auto import tqdm
import time

#os.environ["GROQ_API_KEY"] = ********* (key is cencored for security reasons)


# Initialize the Groq client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

print("Loading and splitting data...")
df = pd.read_csv('../data/train.csv')[['comment_text', 'toxic']].dropna()

# Split the data identically to previous baselines to ensure a fair comparison
X_train, X_test, y_train, y_test = train_test_split(
    df['comment_text'], df['toxic'], test_size=0.2, random_state=42
)

# Sample 100 instances to avoid hitting API rate limits and to speed up local testing
print("Sampling 100 comments for Zero-Shot API testing...")
X_test_sampled = X_test.sample(n=100, random_state=42)
y_test_sampled = y_test.loc[X_test_sampled.index]

Loading and splitting data...
Sampling 100 comments for Zero-Shot API testing...


In [2]:
def classify_comment_with_llm(comment):
    # Prompt Engineering: We provide a specific system instruction to explain the gaming context
    system_prompt = (
        "You are an expert AI moderator for competitive multiplayer games. "
        "Your task is to classify text as either toxic (1) or non-toxic (0). "
        "IMPORTANT: Gaming slang and competitive banter (e.g., 'kill', 'shoot', 'noob', 'destroy') "
        "are normal in gaming contexts and should NOT be flagged as toxic unless accompanied by "
        "real-world threats, severe profanity, or harassment. "
        "Reply ONLY with the number 1 (if toxic) or 0 (if non-toxic). Do not explain."
    )
    
    try:
        # Make the API call to Groq
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Text: \"{comment}\""}
            ],
            model="llama3-8b-8192", # Ultra-fast Llama-3 model hosted on Groq
            temperature=0,          # Set temperature to 0 for deterministic, non-creative outputs
            max_tokens=2            # Restrict output length to just the binary digit
        )
        
        # Extract the model's response and convert it to an integer
        result = chat_completion.choices[0].message.content.strip()
        return int(result) if result in ['0', '1'] else 0
        
    except Exception as e:
        # In case of API timeout or unexpected error, default to Non-Toxic (0)
        return 0

print("Generating predictions via Groq API...")
y_pred = []

# Iterate over the sampled comments with a progress bar
for text in tqdm(X_test_sampled, desc="Calling Llama-3"):
    prediction = classify_comment_with_llm(text)
    y_pred.append(prediction)
    
    # Add a small delay to respect API rate limits
    time.sleep(0.5)

Generating predictions via Groq API...


Calling Llama-3:   0%|          | 0/100 [00:00<?, ?it/s]

In [3]:
print("\n--- ZERO-SHOT LLM (Llama-3) BASELINE RESULTS ---")
# Print the classification report to compare LLM predictions with actual labels
print(classification_report(y_test_sampled, y_pred, target_names=['Non-Toxic', 'Toxic']))


--- ZERO-SHOT LLM (Llama-3) BASELINE RESULTS ---
              precision    recall  f1-score   support

   Non-Toxic       0.88      1.00      0.94        88
       Toxic       0.00      0.00      0.00        12

    accuracy                           0.88       100
   macro avg       0.44      0.50      0.47       100
weighted avg       0.77      0.88      0.82       100



c:\Users\Ayşe Gizem\Desktop\ceng\6. dönem\CENG467-NLP\final\CENG467-Toxicity-Detection-main\CENG467-Toxicity-Detection-main\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ayşe Gizem\Desktop\ceng\6. dönem\CENG467-NLP\final\CENG467-Toxicity-Detection-main\CENG467-Toxicity-Detection-main\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ayşe Gizem\Desktop\ceng\6. dönem\CENG467-NLP\final\CENG467-Toxicity-Detection-main\CENG467-Toxicity-Detection-main\.venv\Lib\site-pac